Linear Data Lab 4

Original lab written by: Emily J. King

Goals: Apply linear combinations of signal (sound) data. Use software to convert the information in (sound) signals, e.g. audio to visualization, audio to spectrogram. Apply linear combinations of (image) signal data. Compute moving averages of images using the Frobenius inner product. Apply basic statistical measures to (sound) signals.

Additional files needed: LinearDatasound.npz, petedog-lab.png, and lineum-lab.png
in the same folder as this ipynb file or in the path. (We have not discussed the path.  If you don't know what "path" means, just make sure the two files are in the same folder.)

NOTE: Due to multiple song files being played, do not use "run all".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio # needed to play audio
from scipy.signal import convolve2d # for two-way moving average

Section 1: Plotting and playing song vectors

Loading the song files.

In [ ]:
npzfile=np.load('LinearDatasound.npz')
locals().update(npzfile)
Fs= 44100 # sampling rate

Song credits:

x = clip from King Gizzard and the Lizard Wizard - Flightless Records - 2017
y = clip from Björk - Hunter - OLI Records - 1998
z = clip from Black Star - Definition - Rawkus Records - 1999
u = clip from Johnny Cash - So Doggone Lonesome - Sun Records - 1955
v = clip from Vicente Fernández - Guadalajara (remasterizado) - 2006

In Python, anything to the right of a hash sign is ignored.  This is called a comment.  If you want to play around with your own sound files, remove the hash signs on the left hand side of each line in the following block of code and modify the file names. 
It's possible you might need to do more on the command line than the two %pip lines below to get these packages working in Jupyter.  If you can't figure it out, then just use the songs provided.
Replace y and yFs with z and zFs, respectively, in the code below to import in two different files and compare them as in code that follows.  Double check that yFs and zFs are both equal to 44100.  If so, set Fs = 44,000. If not, you'll need to change the rate in the audio calls.

In [ ]:
#%pip install audio2numpy # if needed.   
#%pip install ffmpeg # if needed

#import audio2numpy as a2n
#y,yFs=a2n.audio_from_file("YOUR MP3, WAV, or AIFF")
#L=1000001
#y=y[:L,]

Visualize song y.

In [ ]:
plt.plot(y[:,0])

Play song y.

In [ ]:
Audio(y.T, normalize=False, rate=Fs) 

Visualize song z.

In [ ]:
plt.plot(z[:,0])

Listen to song z.

In [ ]:
Audio(z.T, normalize=False, rate=Fs) 

Visualize 0.25 times z.

In [ ]:
plt.plot(0.25*z[:,0])
plt.ylim([np.min(z),np.max(z)])

Listen to 0.25 times z.

In [ ]:
Audio(0.25*z.T, normalize=False, rate=Fs) 

Plot 0.25*z + y.

In [ ]:
plt.plot(0.25*z[:,0]+y[:,0])

Listen to 0.25*z + y.

In [ ]:
Audio(0.25*z.T+y.T, normalize=False, rate=Fs) 

Section 2: Visualize the song vectors using spectrograms.

There are a lot of parameters to make a spectrogram "look good".  We will not cover them in this class.  Just use the parameters below (coupled with Matplotlib's defaults), unless you already have experience with spectrograms.

In [ ]:
Nx = np.size(z,0)
nfft = np.floor(Nx/616).astype(int)
time_length = Nx/Fs

Plot the spectrogram of z.

In [ ]:
fig, ax = plt.subplots()
Pxx, freqs, bins, im = ax.specgram(z[:,0], NFFT=nfft, Fs=Fs)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_xlim(0, time_length)

Plot the spectrogram of y.

In [ ]:
fig, ax = plt.subplots()
Pxx, freqs, bins, im = ax.specgram(y[:,0], NFFT=nfft, Fs=Fs)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_xlim(0, time_length)

Plot the spectrogram of 0.25z+y.

In [ ]:
fig, ax = plt.subplots()
Pxx, freqs, bins, im = ax.specgram(0.25*z[:,0]+y[:,0], NFFT=nfft, Fs=Fs)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_xlim(0, time_length)

Section 3: Linear Combinations of Images

Importing images.  

It's possible to use your own images by changing out the filename.  A few things to note
1. Matplotlib now suggests using a different command, but then you will need to install an additional package. https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.open
In any case, plt.imread still works.  
2. If you want to compute linear combinations of your own images, make sure that they are the same size as each other.  It's possible to crop the images in Python so that they are the same size.  (See, e.g., Lab 1.)

In [ ]:
P=plt.imread('petedog-lab.png')
L=plt.imread('lineum-lab.png')

Visualizing the images.

In [ ]:
plt.imshow(P)

In [ ]:
plt.imshow(L)

Looking at the format of the arrays.

In [ ]:
print("Pete Dog's picture's array shape is",P.shape,"with entries of type",P.dtype)
print("Lineum's picture's array shape is",L.shape,"with entries of type",L.dtype)

The fact that the entries are floats tells us that the values are between 0 and 1.  Let's check.

In [ ]:
print("The maximum value of Pete Dog's picture's array is",np.max(P),"while the minimum is",np.min(P),".")
print("The maximum value of Lineum's picture's array is",np.max(L),"while the minimum is",np.min(L),".")

Let's visualize a linear combination of the images.

In [ ]:
a=0.5
b=0.5
plt.imshow(a*P+b*L)

What happens?  Discuss what you see. Try different values for a and b and look at the output.

Now let's play around with color.

In [ ]:
newP=np.copy(P) # copy is necessary so that the values aren't linked
newP[:,:,2]=2*newP[:,:,2]
plt.imshow(newP)

As a reminder, the original image looked like:

In [ ]:
plt.imshow(P)

What happened?  Discuss what you see.

Let's repeat the same modification.

In [ ]:
newP[:,:,2]=2*newP[:,:,2]
plt.imshow(newP)

Again, discuss what happened.

Section 4: Moving Averages Using the Frobenius Inner Product

We will apply moving average windows of larger and larger size to Pete Dog.  Starting with a 5x5 window.  Note that mode='valid' means that we only go up to the edges of the image without zero-padding, mirroring, or periodically continuing at the edges.  Note also that k**2 is k squared.

In [ ]:
k=5
kernel = np.ones((k,k))/k**2
plt.imshow(convolve2d(grayP, kernel, mode='valid'),cmap=plt.get_cmap('gray'))

Now rerun the code above with k x k windows for k = 11, 33, 99.  What happens?

Section 5: Compute basic statistics of song files.

The average values of z, y, and 0.25z+y.

In [ ]:
print("The average value of song file z is",np.average(z),".")
print("The average value of song file y is",np.average(y),".")
print("The average value of song file 0.25z+z is",np.average(0.25*z+y),".")

The standard deviations of z, y, and 0.25z+y.

In [ ]:
print("The standard deviation of song file z is",np.std(z),".")
print("The standard deviation of song file y is",np.std(y),".")
print("The standard deviation of song file 0.25z+z is",np.std(0.25*z+y),".")

Lab 4 Exercises

1. Choose a pair of songs, two from the files loaded from LinearDatasound.npz, two of your own loaded using the commented out code, or a mixture. Call the song vectors song1 and song2. 

2. Compute and listen to three different linear combinations of the songs.

3. Plot the moving averages of song1 with window lengths of 101 and 1001 using the code below. The ends are trimmed off so that all of the vectors are the same length. (Code provided below.)

In [ ]:
song1_ma101=np.convolve(song1[:,1],np.ones(101)/101,mode='full')
song1_ma1001=np.convolve(song1[:,1],np.ones(1001)/1001,mode='full')

ax = plt.gca()
plt.plot(song1[500:-500,1],label='song1')
plt.plot(song1_ma101[500:-500],label='mov ave 101')
plt.plot(song1_ma1001[500:-500],label='mov ave 1001')
ax.legend()

Now listen to the moving averages.  If you don't hear anything the first time through, turn up your volume and play again.

Note if you use your own songs and the xsr shown during import is not equal to 44,100, then you will need to update the rate in the following commands.

First, play the original song.

In [ ]:
Audio(song1[:,1], normalize=False, rate=44100) 

Next the moving average with window of length 101.

In [ ]:
Audio(song1_ma101[500:-500], normalize=False, rate=44100) 

Finally, the moving average with window of length 1001.

In [ ]:
Audio(song1_ma1001[500:-500], normalize=False, rate=44100) 

Between the plots and the sounds, what are the moving averages doing to the song file?

4. Plot the spectrograms of song1 and song2.

5. a. Modify the Lineum picture to halve the amount of red.

b. Now visualize the new image.

c. Now take the modified image and double the amount of green.

d. Now visualize the new image.

e. Convert the new image to grayscale.

f. Compute and visualize a 11x11 two-way moving average of the grayscale image from part e.